In [2]:
!pip install transformers 

Defaulting to user installation because normal site-packages is not writeable


In [29]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel

V=7
E=64
T=5
#--- Input Sequences (start at 1..T): -----
InSet=torch.tensor([[1, 3, 5, 2, 4], [1, 6, 5, 2, 7]])
#--- Target Sequences: -----
TarSet=torch.tensor([[3, 5, 2, 4, 1],[6, 5, 2, 7, 1]])

print('Inst:',InSet.shape)
print(InSet[0:1,].shape)

#InSet=InSet.unsqueeze(1)
#TarSet=TarSet.unsqueeze(1)
#print('Inst:',InSet.shape)
#print(InSet[0].shape)


Inst: torch.Size([2, 5])
torch.Size([1, 5])


In [20]:
# 1. Setup a Causal Language Model (Auto-regressive)
config = GPT2Config(
    vocab_size=V,  #1000,
    n_embd=E,  #64,
    n_head=1,  #4,
    n_layer=1,  # Single transformer layer
)

print('config:',config)
config.bos_token_id=1
config.eos_token_id=None

config: GPT2Config {
  "activation_function": "gelu_new",
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_embd": 64,
  "n_head": 1,
  "n_inner": null,
  "n_layer": 1,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "transformers_version": "4.56.1",
  "use_cache": true,
  "vocab_size": 7
}



In [22]:
model = GPT2LMHeadModel(config)

# 2. Input Sequence: "The cat sat"
# Shape: (batch_size=1, sequence_length=3)
#input_ids = InSet[0,];  #torch.tensor([[10, 45, 82]])
#input_ids = torch.tensor([[1,2,3]])
print('in ids',input_ids.shape)


# Target Sequence: "cat sat [next_token]"
# We shift the inputs right to create the labels
#labels = TarSet[0:1,];  # torch.tensor([[45, 82, 99]])  # 99 is the token we want to predict next
#labels = torch.tensor([[2,3,1]])
#print('labels',labels.shape)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

in ids torch.Size([1, 5])
labels torch.Size([1, 1, 5])


In [34]:
# 3. Training Loop
model.train()
for step in range(5):
  for input_index in range(2):  #train each input one-at-a-time
      optimizer.zero_grad()
      input_ids = InSet[input_index:input_index+1,];  
      labels   = TarSet[input_index:input_index+1,]; 
      print('in id shape',input_ids.shape, 'indx',input_index)
      print(input_ids)
      # Hugging Face models calculate Cross-Entropy loss internally if 'labels' are provided
      outputs = model(input_ids=input_ids, labels=labels)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      print('done indx',input_index)

  print(f"Step {step+1}, Loss: {loss.item():.4f}")

in id shape torch.Size([1, 5]) indx 0
tensor([[1, 3, 5, 2, 4]])
done indx 0
in id shape torch.Size([1, 5]) indx 1
tensor([[1, 6, 5, 2, 7]])


IndexError: index out of range in self

In [16]:

# 4. Verification (Inference)
model.eval()
with torch.no_grad():
  allpreds=model(input_ids)
  print('all preds',allpreds.logits.shape)

  next_token_logits = model(input_ids).logits[:, -1, :]  # Get predictions for the very last position
  predicted_id = torch.argmax(next_token_logits, dim=-1)
  print(f"\nPredicted next token ID: {predicted_id.item()}")




all preds torch.Size([1, 1, 5, 7])


RuntimeError: a Tensor with 5 elements cannot be converted to Scalar

In [36]:
print(model)


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(7, 64)
    (wpe): Embedding(1024, 64)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): GPT2Block(
        (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=192, nx=64)
          (c_proj): Conv1D(nf=64, nx=64)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=256, nx=64)
          (c_proj): Conv1D(nf=64, nx=256)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=64, out_features=7, bias=False)
)
